# Library: encoding, models, explainers and metrics

Preprocessing, models, explainers and explanation-quality metrics.

Explanation space
-----------------
All explainers operate on the *original* clinical features, not on one-hot
columns. Each dataset is mapped to a numeric matrix Z (one column per original
feature): numeric features are median-imputed, categorical features are
mode-imputed and integer-coded. Models receive Z and do their own internal
encoding (one-hot + scaling for LR/MLP). Attributions therefore always have one
value per clinical feature, which keeps explainers comparable.

In [23]:
from __future__ import annotations
import time
import warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
warnings.filterwarnings("ignore")

## Prep

### `Encoder` (class)

Fit on training rows only; maps raw DataFrame -> numeric matrix Z.

In [24]:
# --------------------------------------------------------------------------- prep
class Encoder:
    """Fit on training rows only; maps raw DataFrame -> numeric matrix Z."""

    def fit(self, X: pd.DataFrame, cat):
        self.cols = list(X.columns)
        self.cat = [c for c in self.cols if c in cat]
        self.cat_idx = [self.cols.index(c) for c in self.cat]
        self.num_idx = [i for i in range(len(self.cols)) if i not in self.cat_idx]
        self.fill, self.levels = {}, {}
        for c in self.cols:
            if c in self.cat:
                s = X[c].astype("object")
                self.fill[c] = s.mode(dropna=True).iloc[0]
                self.levels[c] = sorted(s.dropna().unique().tolist(), key=str)
            else:
                self.fill[c] = float(pd.to_numeric(X[c], errors="coerce").median())
        return self

    def transform(self, X: pd.DataFrame) -> np.ndarray:
        Z = np.zeros((len(X), len(self.cols)))
        for j, c in enumerate(self.cols):
            if c in self.cat:
                s = X[c].astype("object").where(X[c].notna(), self.fill[c])
                m = {v: k for k, v in enumerate(self.levels[c])}
                Z[:, j] = s.map(lambda v: m.get(v, m[self.fill[c]])).astype(float).values
            else:
                Z[:, j] = pd.to_numeric(X[c], errors="coerce").fillna(self.fill[c]).values
        return Z

    def baseline(self, Ztr):
        """Reference value per feature used by deletion metrics (median / mode)."""
        b = np.median(Ztr, axis=0)
        for j in self.cat_idx:
            vals, cnt = np.unique(Ztr[:, j], return_counts=True)
            b[j] = vals[cnt.argmax()]
        return b

## Models

### `_ct` (function)

In [25]:
# --------------------------------------------------------------------------- models
def _ct(enc):
    tr = []
    if enc.num_idx:
        tr.append(("num", StandardScaler(), enc.num_idx))
    if enc.cat_idx:
        tr.append(("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), enc.cat_idx))
    return ColumnTransformer(tr)

### `make_model` (function)

In [26]:
def make_model(kind, enc, seed=0, params=None):
    params = params or {}
    if kind == "LR":
        return Pipeline([("ct", _ct(enc)), ("m", LogisticRegressionCV(Cs=10, cv=5, max_iter=5000,
                                                                      scoring="roc_auc", random_state=seed))])
    if kind == "MLP":
        p = dict(hidden_layer_sizes=(64, 32), alpha=1e-3, learning_rate_init=1e-3)
        p.update(params)
        return Pipeline([("ct", _ct(enc)), ("m", MLPClassifier(max_iter=600, early_stopping=True,
                                                               n_iter_no_change=20, random_state=seed, **p))])
    if kind == "XGB":
        from xgboost import XGBClassifier
        p = dict(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                 min_child_weight=1, reg_lambda=1.0)
        p.update(params)
        return XGBClassifier(eval_metric="logloss", random_state=seed, n_jobs=2, tree_method="hist", **p)
    if kind == "EBM":
        from interpret.glassbox import ExplainableBoostingClassifier
        ft = ["nominal" if j in enc.cat_idx else "continuous" for j in range(len(enc.cols))]
        return ExplainableBoostingClassifier(interactions=0, outer_bags=8, feature_types=ft,
                                             random_state=seed, n_jobs=1)
    raise ValueError(kind)

In [27]:
TUNING = {
    "XGB": dict(n_estimators=[100, 200, 300, 500], max_depth=[2, 3, 4, 5], learning_rate=[0.02, 0.05, 0.1],
                min_child_weight=[1, 3, 5], subsample=[0.7, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0]),
    "MLP": {"m__alpha": [1e-4, 1e-3, 1e-2, 1e-1], "m__hidden_layer_sizes": [(32,), (64, 32), (128, 64)],
            "m__learning_rate_init": [1e-3, 3e-3]},
}

### `tune` (function)

Randomised search with 3-fold stratified CV on the training split only.

In [28]:
def tune(kind, enc, Z, y, seed=0, n_iter=12):
    """Randomised search with 3-fold stratified CV on the training split only."""
    if kind not in TUNING:
        return {}
    base = make_model(kind, enc, seed)
    rs = RandomizedSearchCV(base, TUNING[kind], n_iter=n_iter, scoring="roc_auc", n_jobs=1, random_state=seed,
                            cv=StratifiedKFold(3, shuffle=True, random_state=seed))
    rs.fit(Z, y)
    best = rs.best_params_
    if kind == "MLP":
        best = {k.replace("m__", ""): v for k, v in best.items()}
    return best

### `proba_fn` (function)

In [29]:
def proba_fn(model):
    return lambda Z: model.predict_proba(np.asarray(Z, dtype=float))[:, 1]

## Explainers

### `explain_kernelshap` (function)

In [30]:
# --------------------------------------------------------------------------- explainers
def explain_kernelshap(f, Zbg, Zx, nsamples=None, seed=0):
    import shap
    M = Zx.shape[1]
    ns = nsamples or min(2 * M + 2048, 600)
    ex = shap.KernelExplainer(f, Zbg, link="identity", seed=seed)
    return np.asarray(ex.shap_values(Zx, nsamples=ns, silent=True))

### `explain_lime` (function)

In [31]:
def explain_lime(f, Ztr, Zx, cols, cat_idx, seed=0, num_samples=1000):
    from lime.lime_tabular import LimeTabularExplainer
    ex = LimeTabularExplainer(Ztr, feature_names=cols, categorical_features=cat_idx, mode="classification",
                              discretize_continuous=True, random_state=seed)
    f2 = lambda Z: np.column_stack([1 - f(Z), f(Z)])
    M = Zx.shape[1]
    out = np.zeros_like(Zx, dtype=float)
    for i, x in enumerate(Zx):
        e = ex.explain_instance(x, f2, num_features=M, num_samples=num_samples, labels=(1,))
        for j, w in e.as_map()[1]:
            out[i, j] = w
    return out

### `explain_treeshap` (function)

In [32]:
def explain_treeshap(model, Zx):
    import shap
    ex = shap.TreeExplainer(model)  # path-dependent; output in log-odds
    return np.asarray(ex.shap_values(Zx))

### `explain_ebm` (function)

Exact additive term contributions of a main-effects-only EBM (log-odds).

In [33]:
def explain_ebm(model, Zx):
    """Exact additive term contributions of a main-effects-only EBM (log-odds)."""
    ex = model.eval_terms(Zx)
    return np.asarray(ex)

### `explain_linear` (function)

Exact interventional SHAP for logistic regression in log-odds space:
sum over a feature's encoded columns of w_k * (h_k(x) - E[h_k]).

In [34]:
def explain_linear(model, enc, Ztr, Zx):
    """Exact interventional SHAP for logistic regression in log-odds space:
    sum over a feature's encoded columns of w_k * (h_k(x) - E[h_k])."""
    ct, lr = model.named_steps["ct"], model.named_steps["m"]
    Htr, Hx = ct.transform(Ztr), ct.transform(Zx)
    contrib = (Hx - Htr.mean(axis=0)) * lr.coef_[0]
    # map encoded columns back to original features
    owner = []
    for name, tr, idx in ct.transformers_:
        if name == "num":
            owner += list(idx)
        elif name == "cat":
            for j, cats in zip(idx, tr.categories_):
                owner += [j] * len(cats)
    owner = np.array(owner)
    out = np.zeros_like(Zx, dtype=float)
    for j in range(Zx.shape[1]):
        out[:, j] = contrib[:, owner == j].sum(axis=1)
    return out

### `explain_random` (function)

In [35]:
def explain_random(Zx, seed=0):
    return np.random.default_rng(seed).normal(size=Zx.shape)

In [36]:
NATIVE = {"LR": "Exact-linear", "EBM": "EBM-native", "XGB": "TreeSHAP", "MLP": None}

### `explain_all` (function)

In [37]:
def explain_all(kind, model, enc, Ztr, Zbg, Zx, seed=0, which=None):
    f = proba_fn(model)
    res, t = {}, {}
    which = which or ["KernelSHAP", "LIME", "native", "Random"]
    if "KernelSHAP" in which:
        s = time.time(); res["KernelSHAP"] = explain_kernelshap(f, Zbg, Zx, seed=seed); t["KernelSHAP"] = time.time() - s
    if "LIME" in which:
        s = time.time(); res["LIME"] = explain_lime(f, Ztr, Zx, enc.cols, enc.cat_idx, seed=seed); t["LIME"] = time.time() - s
    if "native" in which and NATIVE[kind]:
        s = time.time()
        if kind == "XGB":
            a = explain_treeshap(model, Zx)
        elif kind == "EBM":
            a = explain_ebm(model, Zx)
        else:
            a = explain_linear(model, enc, Ztr, Zx)
        res[NATIVE[kind]] = a; t[NATIVE[kind]] = time.time() - s
    if "Random" in which:
        res["Random"] = explain_random(Zx, seed); t["Random"] = 0.0
    return res, t

## Metrics

### `deletion_curves` (function)

Most-relevant-first (MoRF) and least-relevant-first (LeRF) deletion.
Features are replaced, in order of |attribution|, by the training reference value.
Returns per-instance normalised areas under |f(x_k) - f(x)| (range 0-1).

In [38]:
# --------------------------------------------------------------------------- metrics
def deletion_curves(f, Zx, A, base):
    """Most-relevant-first (MoRF) and least-relevant-first (LeRF) deletion.
    Features are replaced, in order of |attribution|, by the training reference value.
    Returns per-instance normalised areas under |f(x_k) - f(x)| (range 0-1)."""
    n, M = Zx.shape
    p0 = f(Zx)
    out = {}
    for name, order_sign in (("MoRF", -1), ("LeRF", 1)):
        order = np.argsort(order_sign * np.abs(A), axis=1, kind="stable")
        Zk = Zx.copy()
        curve = np.zeros((n, M))
        for k in range(M):
            idx = order[:, k]
            Zk[np.arange(n), idx] = base[idx]
            curve[:, k] = np.abs(f(Zk) - p0)
        out[name] = curve.mean(axis=1)
    return out["MoRF"], out["LeRF"]

### `topk_sets` (function)

In [39]:
def topk_sets(A, k):
    return np.argsort(-np.abs(A), axis=1)[:, :k]

### `rank_corr` (function)

In [40]:
def rank_corr(a, b):
    r = spearmanr(np.abs(a), np.abs(b)).correlation
    return 0.0 if np.isnan(r) else r

### `pairwise_agreement` (function)

Per instance: top-k feature agreement, top-k sign agreement, Spearman of |attr|.

In [41]:
def pairwise_agreement(A, B, k=5):
    """Per instance: top-k feature agreement, top-k sign agreement, Spearman of |attr|."""
    n = A.shape[0]
    ta, tb = topk_sets(A, k), topk_sets(B, k)
    fa, sa, rc = np.zeros(n), np.zeros(n), np.zeros(n)
    for i in range(n):
        inter = set(ta[i]) & set(tb[i])
        fa[i] = len(inter) / k
        sa[i] = sum(np.sign(A[i, j]) == np.sign(B[i, j]) for j in inter) / k
        rc[i] = rank_corr(A[i], B[i])
    return fa, sa, rc

### `stability` (function)

Per-instance Spearman of |attr| and top-k Jaccard between two attribution sets.

In [42]:
def stability(A, B, k=5):
    """Per-instance Spearman of |attr| and top-k Jaccard between two attribution sets."""
    n = A.shape[0]
    ta, tb = topk_sets(A, k), topk_sets(B, k)
    sp = np.array([rank_corr(A[i], B[i]) for i in range(n)])
    jac = np.array([len(set(ta[i]) & set(tb[i])) / len(set(ta[i]) | set(tb[i])) for i in range(n)])
    return sp, jac

### `entropy` (function)

In [43]:
def entropy(A):
    P = np.abs(A) + 1e-12
    P = P / P.sum(axis=1, keepdims=True)
    return -(P * np.log(P)).sum(axis=1) / np.log(A.shape[1])

### `perturb` (function)

In [44]:
def perturb(Zx, enc, Ztr, scale=0.05, rng=None):
    rng = rng or np.random.default_rng(0)
    Zp = Zx.copy()
    sd = Ztr.std(axis=0)
    for j in enc.num_idx:
        if len(np.unique(Ztr[:, j])) <= 5:  # binary / short ordinal codes are left unchanged
            continue
        Zp[:, j] = Zx[:, j] + rng.normal(0, scale * sd[j], size=len(Zx))
    return Zp